# Tutorial 2 -- Recipes

A **recipe** is a high-level pipeline assembled once from ops, parameterised by a small Pydantic config. Recipes cover the typical wind-tunnel post-processing flows:

| Recipe | What it does |
|---|---|
| `build_cp` | $(p - p_{ref}) / q_\infty$ with optional time rescale and trailing statistics |
| `cf_pipeline` | per-body, per-direction Cf via area-weighted mean of Cp |
| `cm_pipeline` | per-body, per-direction Cm via sum of moment contributions |
| `ce_pipeline` | per-zone Ce via area-weighted mean of Cp |
| `build_s1` | velocity-profile ratio against a reference profile |
| `build_pedestrian_comfort` | probe-extract velocity + per-probe statistics |
| `build_dynamic_response` | physical loads -> modal solver -> physical response |

This notebook walks a Cp -> Cf chain. Other recipes follow the same shape: build the config, call the recipe.

## 1. Synthesise a body pressure data source

Five triangles, two bodies, 500 timesteps. In a real run this comes from `XdmfH5Storage.read_data_source("bodies.cp")`.

In [1]:
import numpy as np
from cfdmod import (
    SurfaceDataSource,
    TimeAxis,
    Topology,
    ElementMeta,
    Grouping,
    MemoryFieldStore,
)

rng = np.random.default_rng(42)
n_elements, n_t = 5, 500

# Mesh: 5 triangles arranged into 2 "bodies".
vertices = np.array([[i, j, 0] for i in range(3) for j in range(3)], dtype=np.float64)
tris = np.array(
    [
        [0, 1, 3],
        [1, 4, 3],  # body 0
        [4, 5, 7],
        [5, 8, 7],
        [4, 7, 6],  # body 1
    ],
    dtype=np.int32,
)

# Synthetic pressure: body 0 at ~110 Pa, body 1 at ~95 Pa.
p = np.empty((n_elements, n_t))
p[:2] = rng.normal(110.0, 5.0, size=(2, n_t))
p[2:] = rng.normal(95.0, 5.0, size=(3, n_t))

body_ds = SurfaceDataSource(
    time=TimeAxis(initial_time=0.0, timestep_size=0.001, n_timesteps=n_t),
    topology=Topology.triangles(tris, vertices),
    elements=ElementMeta(area=np.array([0.5, 0.5, 0.5, 0.5, 0.5])),
    groupings={"body": Grouping(name="body", indices=[0, 0, 1, 1, 1])},
    fields=MemoryFieldStore({"pressure": p}),
)
print(
    "body_ds:", body_ds.kind, body_ds.n_elements, "triangles,", body_ds.time.n_timesteps, "steps"
)

body_ds: surface 5 triangles, 500 steps


## 2. Build Cp with a scalar reference and dynamic pressure

`CpRecipeConfig` declares the recipe inputs. The recipe is a `Pipeline = Callable[[DataSource], DataSource]`. Run it by calling it on the body.

In [2]:
from cfdmod.recipes import CpRecipeConfig, build_cp

p_ref = 100.0  # static reference, Pa
q_inf = 0.5 * 1.225 * 12.0**2  # 0.5 * rho * U^2

cp_ds = build_cp(
    body_ds,
    p_ref=p_ref,
    cfg=CpRecipeConfig(dynamic_pressure=q_inf, out="cp"),
)
cp = cp_ds.fields.read("cp")
print(f"cp shape:    {cp.shape}")
print(f"cp body 0 mean: {cp[:2].mean():.3f}")
print(f"cp body 1 mean: {cp[2:].mean():.3f}")

cp shape:    (5, 500)
cp body 0 mean: 0.112
cp body 1 mean: -0.060


### Statistics in the same recipe

Setting `statistics=...` collapses the time axis after the Cp computation. The result is a time-aggregated `SurfaceDataSource` (one row per triangle, one column per requested stat).

In [3]:
cp_stats = build_cp(
    body_ds,
    p_ref=p_ref,
    cfg=CpRecipeConfig(
        dynamic_pressure=q_inf,
        statistics=["mean", "rms", "peak_min", "peak_max"],
    ),
)
print("time aggregated:", cp_stats.time.is_time_aggregated)
print("stat fields:    ", cp_stats.field_names)
for name in cp_stats.field_names:
    arr = cp_stats.fields.read(name)
    print(f"  {name:>8}  per-element: {arr}")

time aggregated: True
stat fields:     ['mean', 'rms', 'peak_min', 'peak_max']
      mean  per-element: [ 0.11263457  0.11084711 -0.05691174 -0.06569352 -0.05675177]
       rms  per-element: [0.05441803 0.05773108 0.05789709 0.0568469  0.05782577]
  peak_min  per-element: [-0.03212349 -0.09344744 -0.22424143 -0.22945763 -0.23037301]
  peak_max  per-element: [0.27856363 0.29358581 0.10256297 0.10851727 0.10529696]


## 3. Run Cf on top of the time-resolved Cp

Cf is per-body, per-direction. Synthesise direction-tagged Cp fields (cp_x / cp_y / cp_z) -- in production these come from the projection onto each body's normals.

In [4]:
from cfdmod.recipes import CfRecipeConfig, cf_pipeline

# Pretend the body Cp acts in three orthogonal directions for the demo.
cp_t = cp_ds.fields.read("cp")
cf_input = (
    cp_ds.with_field("cp_x", cp_t * 0.7)
    .with_field("cp_y", cp_t * 0.2)
    .with_field("cp_z", cp_t * 0.1)
)

cf = cf_pipeline(CfRecipeConfig(grouping="body", directions=["x", "y", "z"]))(cf_input)
print("kind:       ", cf.kind, "(GroupsDataSource)")
print("n_elements: ", cf.n_elements, "(= number of bodies)")
for f in ["cf_x", "cf_y", "cf_z"]:
    print(f"  {f} mean per body: {cf.fields.read(f).mean(axis=1)}")

kind:        groups (GroupsDataSource)
n_elements:  2 (= number of bodies)
  cf_x mean per body: [ 0.07821859 -0.04184997]
  cf_y mean per body: [ 0.02234817 -0.01195714]
  cf_z mean per body: [ 0.01117408 -0.00597857]


## 4. Compose your own chain

Recipes are just `compose(...)` of ops. For one-off variants, drop down to the ops layer:

In [5]:
from cfdmod import compose
from cfdmod.core.ops.field import moving_average, MovingAverageParams
from cfdmod.core.ops.data_source_create import compute_statistics, StatisticsParams
from functools import partial

smooth_then_stats = compose(
    partial(moving_average, p=MovingAverageParams(field="cp", window=0.05)),
    partial(compute_statistics, p=StatisticsParams(field="cp", kinds=["mean", "rms"])),
)
smoothed_stats = smooth_then_stats(cp_ds)
print("smoothed stats fields:", smoothed_stats.field_names)
print("mean per triangle:    ", smoothed_stats.fields.read("mean"))

smoothed stats fields: ['mean', 'rms']
mean per triangle:     [ 0.11199655  0.11218037 -0.05755447 -0.0645187  -0.05638211]


## Next: Tutorial 3 -- Pipelines via YAML template